# Notebook 01 — Pose Exploration

**Goal:** Run MediaPipe Pose on a video clip, visualise the keypoints, and understand the data format before writing any model code.

**This notebook is your sandbox** — break things freely here. Once something works, port the clean version to `module1/pose/extractor.py`.

### Steps
1. Load a sample video clip
2. Extract keypoints with MediaPipe
3. Visualise the skeleton on a frame
4. Inspect the data shapes and values
5. Save keypoints to `data/processed/`

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys
sys.path.append('..')   # so we can import from the project root

import cv2
import mediapipe as mp
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

print('Imports OK')

In [ ]:
# ── Load a single frame from a video ─────────────────────────────────────────
VIDEO_PATH = '../data/samples/clip.mp4'   # put your clip here

cap = cv2.VideoCapture(VIDEO_PATH)
ret, frame = cap.read()
cap.release()

if not ret:
    print('Could not read video. Check VIDEO_PATH.')
else:
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(12, 7))
    plt.imshow(frame_rgb)
    plt.title('First frame')
    plt.axis('off')
    plt.show()
    print(f'Frame shape: {frame.shape}')  # (H, W, 3)

In [ ]:
# ── Run MediaPipe Pose on the frame ──────────────────────────────────────────
mp_pose    = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    results = pose.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

if results.pose_landmarks:
    # Draw skeleton on a copy of the frame
    annotated = frame.copy()
    mp_drawing.draw_landmarks(annotated, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
    plt.figure(figsize=(12, 7))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.title('MediaPipe Pose — 33 keypoints')
    plt.axis('off')
    plt.show()
else:
    print('No person detected in this frame. Try a different clip.')

In [ ]:
# ── Inspect keypoint values ───────────────────────────────────────────────────
if results.pose_landmarks:
    keypoints = np.array([
        [lm.x, lm.y, lm.z, lm.visibility]
        for lm in results.pose_landmarks.landmark
    ])
    print(f'Keypoints shape: {keypoints.shape}')  # (33, 4)
    print(f'\nFirst 5 joints (x, y, z, visibility):')
    print(keypoints[:5])

    from module1.pose.extractor import KEYPOINT_NAMES
    print(f'\nKey joint names:')
    for i in [11, 12, 23, 24, 25, 26, 27, 28]:   # shoulders, hips, knees, ankles
        x, y, z, vis = keypoints[i]
        print(f'  [{i:2d}] {KEYPOINT_NAMES[i]:<20} x={x:.3f}  y={y:.3f}  vis={vis:.3f}')

In [ ]:
# ── Use the project extractor to process the full video ──────────────────────
from module1.pose.extractor import PoseExtractor, save_keypoints

with PoseExtractor() as extractor:
    all_keypoints = extractor.extract_from_video(VIDEO_PATH)

print(f'Extracted keypoints shape: {all_keypoints.shape}')   # (n_frames, 33, 4)
save_keypoints(all_keypoints, '../data/processed/', 'sample_clip')

In [ ]:
# ── Plot how one joint moves across the video ─────────────────────────────────
# Joint 27 = left ankle, joint 28 = right ankle
left_ankle_y  = all_keypoints[:, 27, 1]   # y-coordinate over time
right_ankle_y = all_keypoints[:, 28, 1]

plt.figure(figsize=(14, 4))
plt.plot(left_ankle_y,  label='Left ankle (y)', color='steelblue')
plt.plot(right_ankle_y, label='Right ankle (y)', color='tomato')
plt.xlabel('Frame')
plt.ylabel('Normalised y-position')
plt.title('Ankle y-position over time — early indicator of stride asymmetry')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()